02_feature_extraction.ipynb
Purpose:

Extract all acoustic features
Save to SQLite

In [1]:
import sqlite3
from pathlib import Path

import librosa
import numpy as np
import pandas as pd

In [6]:
class Wav:
    """Audio feature extraction class"""

    def get_paths(self, query, db_path):
        """Load all wav paths from database"""

        with sqlite3.connect(db_path) as conn:
            df = pd.read_sql_query(query, conn)

        return df["absolute_path"].tolist()

    def extract_features(self, path):
        """Extract all acoustic features from one wav file"""

        y, sr = librosa.load(path, sr=None)

        spectral_rolloff = librosa.feature.spectral_rolloff(
            y=y,
            sr=sr
        )[0]

        # Duration
        duration = librosa.get_duration(y=y, sr=sr)

        # Pitch
        pitch = librosa.yin(
            y,
            fmin=50,
            fmax=400
        )

        pitch = pitch[np.isfinite(pitch)]

        # Energy (RMS)
        rms = librosa.feature.rms(y=y)[0]

        # Spectral features
        spectral_centroid = librosa.feature.spectral_centroid(
            y=y,
            sr=sr
        )[0]

        spectral_bandwidth = librosa.feature.spectral_bandwidth(
            y=y,
            sr=sr
        )[0]

        intervals = librosa.effects.split(
            y,
            top_db=20
        )

        speech_samples = sum(
            end - start
            for start, end in intervals
        )
        speech_duration = speech_samples / sr
        silence_ratio = (
            duration - speech_duration
        ) / duration
        if duration > 0:
            silence_ratio = (
                duration - speech_duration
            ) / duration
        else:
            silence_ratio = 0

        spectral_contrast = librosa.feature.spectral_contrast(
            y=y,
            sr=sr
        )
        spectral_contrast_mean = np.mean(spectral_contrast)

        # Zero Crossing Rate
        zcr = librosa.feature.zero_crossing_rate(y)[0]

        pitch = pitch[np.isfinite(pitch)]

        if len(pitch) == 0:
            pitch_mean = np.nan
            pitch_std = np.nan
            pitch_min = np.nan
            pitch_max = np.nan
            pitch_range = np.nan
        else:
            pitch_mean = np.mean(pitch)
            pitch_std = np.std(pitch)
            pitch_min = np.min(pitch)
            pitch_max = np.max(pitch)
            pitch_range = pitch_max - pitch_min

        return {
            "file": Path(path).name,

            "duration": duration,

            "spectral_rolloff": np.mean(spectral_rolloff),

            "spectral_contrast": spectral_contrast_mean,

            "silence_ratio": silence_ratio,

            "pitch_mean": pitch_mean,
            "pitch_std": pitch_std,
            "pitch_min": pitch_min,
            "pitch_max": pitch_max,
            "pitch_range": pitch_range,

            "energy_mean": np.mean(rms),
            "energy_std": np.std(rms),
            "energy_range": np.max(rms) - np.min(rms),

            "spectral_centroid": np.mean(spectral_centroid),

            "spectral_bandwidth": np.mean(spectral_bandwidth),

            "zcr": np.mean(zcr)
        }

    def save_to_database(self, data, db_path):
        """Save dataframe to SQLite"""

        df = pd.DataFrame(data)

        with sqlite3.connect(db_path) as conn:
            df.to_sql(
                name="wav_attribute",
                con=conn,
                if_exists="replace",
                index=False
            )

        print(f"Successfully saved {len(df)} records")

        return df

Run Extraction

In [7]:
DB_PATH = "../data/processed/system_data.db"

wav = Wav()

audio_paths = wav.get_paths(
    "SELECT absolute_path FROM wav_path",
    DB_PATH
)

In [8]:
if __name__ == "__main__":

    DB_PATH = "../data/processed/system_data.db"

    wav = Wav()

    audio_paths = wav.get_paths(
        query="SELECT absolute_path FROM wav_path",
        db_path=DB_PATH
    )

    results = []

    total = len(audio_paths)

    for idx, path in enumerate(audio_paths, start=1):

        try:
            features = wav.extract_features(path)

            results.append(features)

            if idx % 100 == 0:
                print(f"Processed {idx}/{total}")

        except Exception as e:
            print(f"Error: {path}")
            print(e)

    wav.save_to_database(
        results,
        DB_PATH
    )

    print("Feature extraction completed.")

Processed 100/3382
Processed 200/3382
Processed 300/3382
Processed 400/3382
Processed 500/3382
Processed 600/3382
Processed 700/3382
Processed 800/3382
Processed 900/3382
Processed 1000/3382
Processed 1100/3382
Processed 1200/3382
Processed 1300/3382
Processed 1400/3382
Processed 1500/3382
Processed 1600/3382
Processed 1700/3382
Processed 1800/3382
Processed 1900/3382
Processed 2000/3382
Processed 2100/3382
Processed 2200/3382
Processed 2300/3382
Processed 2400/3382
Processed 2500/3382
Processed 2600/3382
Processed 2700/3382
Processed 2800/3382
Processed 2900/3382
Processed 3000/3382
Processed 3100/3382
Processed 3200/3382
Processed 3300/3382
Successfully saved 3382 records
Feature extraction completed.
